# 常见的大模型优化方法
* KV Cache
* attention变体：（MQA/GQA）
* Flash Attention

## KV Cache
Q1*K1 ~~Q1*K2~~

Q2*K1 Q2*K2 

... ... 

省去右上角以及上一行的运算

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(42)


class SimpleSelfAttention(nn.Module):
    def __init__(self, hidden_size=16, num_heads=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def split_heads(self, x):
        # x: [B, T, H]
        B, T, H = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        # [B, num_heads, T, head_dim]

    def merge_heads(self, x):
        # x: [B, num_heads, T, head_dim]
        B, nh, T, hd = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, self.hidden_size)
        # [B, T, H]

    def forward_no_cache(self, x):
        """
        无 KV Cache：
        每次输入完整序列 x，重新计算全部 Q/K/V。
        """
        Q = self.split_heads(self.q_proj(x))
        K = self.split_heads(self.k_proj(x))
        V = self.split_heads(self.v_proj(x))

        attn_score = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)

        T = x.size(1)
        causal_mask = torch.tril(torch.ones(T, T, device=x.device))
        attn_score = attn_score.masked_fill(causal_mask == 0, float("-inf"))

        attn_prob = F.softmax(attn_score, dim=-1)
        out = attn_prob @ V

        out = self.merge_heads(out)
        out = self.o_proj(out)
        return out

    def forward_with_cache(self, x_new, past_k=None, past_v=None):
        """
        有 KV Cache：
        x_new 通常只有一个 token: [B, 1, H]

        只计算当前 token 的 Q/K/V。
        历史 K/V 从 cache 里拿。
        """
        Q_new = self.split_heads(self.q_proj(x_new))
        K_new = self.split_heads(self.k_proj(x_new))
        V_new = self.split_heads(self.v_proj(x_new))

        if past_k is None:
            K_all = K_new
            V_all = V_new
        else:
            K_all = torch.cat([past_k, K_new], dim=2)
            V_all = torch.cat([past_v, V_new], dim=2)

        attn_score = Q_new @ K_all.transpose(-2, -1) / (self.head_dim ** 0.5)
        attn_prob = F.softmax(attn_score, dim=-1)
        out = attn_prob @ V_all

        out = self.merge_heads(out)
        out = self.o_proj(out)

        return out, K_all, V_all

In [3]:
B = 1
T = 5
H = 16

model = SimpleSelfAttention(hidden_size=H, num_heads=4)

x = torch.randn(B, T, H)

# 方式1：无 KV Cache，一次性算完整序列
full_out = model.forward_no_cache(x)
last_full_out = full_out[:, -1:, :]

# 方式2：有 KV Cache，一个 token 一个 token 地喂
past_k, past_v = None, None
cached_out = None

for t in range(T):
    x_new = x[:, t:t + 1, :]
    cached_out, past_k, past_v = model.forward_with_cache(
        x_new,
        past_k,
        past_v,
    )

    print(f"step {t + 1}")
    print("x_new:", x_new.shape)
    print("K cache:", past_k.shape)
    print("V cache:", past_v.shape)
    print("out:", cached_out.shape)
    print("-" * 40)

# 最后一个 token 的输出应该一致
print("无 Cache 最后输出:", last_full_out[0, 0, :5])
print("有 Cache 最后输出:", cached_out[0, 0, :5])

assert torch.allclose(last_full_out, cached_out, atol=1e-5)

print("测试通过：KV Cache 和无 Cache 的最后 token 输出一致。")

step 1
x_new: torch.Size([1, 1, 16])
K cache: torch.Size([1, 4, 1, 4])
V cache: torch.Size([1, 4, 1, 4])
out: torch.Size([1, 1, 16])
----------------------------------------
step 2
x_new: torch.Size([1, 1, 16])
K cache: torch.Size([1, 4, 2, 4])
V cache: torch.Size([1, 4, 2, 4])
out: torch.Size([1, 1, 16])
----------------------------------------
step 3
x_new: torch.Size([1, 1, 16])
K cache: torch.Size([1, 4, 3, 4])
V cache: torch.Size([1, 4, 3, 4])
out: torch.Size([1, 1, 16])
----------------------------------------
step 4
x_new: torch.Size([1, 1, 16])
K cache: torch.Size([1, 4, 4, 4])
V cache: torch.Size([1, 4, 4, 4])
out: torch.Size([1, 1, 16])
----------------------------------------
step 5
x_new: torch.Size([1, 1, 16])
K cache: torch.Size([1, 4, 5, 4])
V cache: torch.Size([1, 4, 5, 4])
out: torch.Size([1, 1, 16])
----------------------------------------
无 Cache 最后输出: tensor([-0.0341, -0.1729, -0.0441, -0.2067, -0.2312], grad_fn=<SliceBackward0>)
有 Cache 最后输出: tensor([-0.0341, -0.17

## attention变体

### MHA 
QKV一一对应
### MQA
只有一个KV
### GQA
一组Q共享一个KV，有一个复制头的动作


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleGQA(nn.Module):
    def __init__(self, hidden_size=16, num_q_heads=4, num_kv_heads=2):
        super().__init__()
        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads
        self.repeat = num_q_heads // num_kv_heads

        self.q_proj = nn.Linear(hidden_size, num_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def split_q(self, x):
        # [B, T, H] -> [B, q_heads, T, head_dim]
        B, T, _ = x.shape
        return x.view(B, T, self.num_q_heads, self.head_dim).transpose(1, 2)

    def split_kv(self, x):
        # [B, T, kv_heads * head_dim] -> [B, kv_heads, T, head_dim]
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def repeat_kv(self, x):
        # [B, kv_heads, T, head_dim]
        # -> [B, q_heads, T, head_dim]
        B, kv_heads, T, D = x.shape
        x = x[:, :, None, :, :]               # [B, kv_heads, 1, T, D]
        x = x.expand(B, kv_heads, self.repeat, T, D)
        return x.reshape(B, kv_heads * self.repeat, T, D)

    def forward(self, x):
        # x: [B, T, H]
        B, T, H = x.shape

        q = self.split_q(self.q_proj(x))
        k = self.split_kv(self.k_proj(x))
        v = self.split_kv(self.v_proj(x))

        print("q:", q.shape)
        print("k before repeat:", k.shape)
        print("v before repeat:", v.shape)

        k = self.repeat_kv(k)
        v = self.repeat_kv(v)

        print("k after repeat:", k.shape)
        print("v after repeat:", v.shape)

        attn_score = q @ k.transpose(-2, -1) / (self.head_dim ** 0.5)
        print("attn_score:", attn_score.shape)

        mask = torch.tril(torch.ones(T, T, device=x.device))
        attn_score = attn_score.masked_fill(mask == 0, float("-inf"))

        attn_prob = F.softmax(attn_score, dim=-1)
        out = attn_prob @ v

        print("attn out before merge:", out.shape)

        out = out.transpose(1, 2).contiguous().view(B, T, H)
        out = self.o_proj(out)

        print("final out:", out.shape)
        return out


if __name__ == "__main__":
    torch.manual_seed(42)

    B = 1
    T = 5
    H = 16

    x = torch.randn(B, T, H)

    model = SimpleGQA(
        hidden_size=16,
        num_q_heads=4,
        num_kv_heads=2,
    )

    y = model(x)

q: torch.Size([1, 4, 5, 4])
k before repeat: torch.Size([1, 2, 5, 4])
v before repeat: torch.Size([1, 2, 5, 4])
k after repeat: torch.Size([1, 4, 5, 4])
v after repeat: torch.Size([1, 4, 5, 4])
attn_score: torch.Size([1, 4, 5, 5])
attn out before merge: torch.Size([1, 4, 5, 4])
final out: torch.Size([1, 5, 16])


## Flash Attention
### V1
分块（Tiling）+ 重计算
### V2
V1 已经解决了 Attention 的 IO 瓶颈，V2 主要解决 GPU 并行度不足的问题。通过更细粒度的 work partition、sequence dimension parallelism、更高效的 warp 调度和 kernel 融合，使 Tensor Core 利用率显著提升，在保持相同算法和显存占用的情况下获得接近 2 倍的训练速度提升。